# GSM8K pass@1 plots

This notebook pulls GSM8K runs from W&B project `ai2-llm/open_instruct_internal`, extracts selected metrics from those runs, normalizes eval steps to 100-step bins, and plots pass@1 over `Step` for the requested DDP active-learning groups. The initial point at `Step == 0` comes from run `qtenf0on` and is recorded with run name `initial_eval`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import wandb

try:
    import seaborn as sns
except ImportError as exc:
    raise ImportError("This notebook requires seaborn. Install it in the active environment before running.") from exc

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [ ]:
PROJECT_PATH = "ai2-llm/open_instruct_internal"
METRICS = {
    "pass_at_1": "eval/pass_at_1",
}
PLOT_METRIC_NAME = "pass_at_1"
INITIAL_RUN_ID = "qtenf0on"
INITIAL_RUN_NAME = "initial_eval"

GROUPS = [
    "ddp_active_n8_k32",
    "ddp_active_n16_k16",
    "ddp_active_n32_k8",
    "ddp_active_n64_k4",
]
GROUP_LABELS = {
    "ddp_active_n8_k32": "N=8, K=32",
    "ddp_active_n16_k16": "N=16, K=16",
    "ddp_active_n32_k8": "N=32, K=8",
    "ddp_active_n64_k4": "N=64, K=4",
}

STEP_BIN = 100
MAX_STEP = 2000
REPO_OR_NOTEBOOK_DIR = Path.cwd()
OUTPUT_DIR = (REPO_OR_NOTEBOOK_DIR if REPO_OR_NOTEBOOK_DIR.name == "notebooks" else REPO_OR_NOTEBOOK_DIR / "notebooks") / "figures"

api = wandb.Api(timeout=90)

## Pull W&B runs

In [ ]:
def normalized_step(step: int | float, *, bin_size: int = STEP_BIN) -> int:
    return int(round(float(step) / bin_size) * bin_size)


def collect_group_runs(groups: list[str]) -> tuple[pd.DataFrame, dict[str, wandb.apis.public.Run]]:
    rows = []
    run_by_id = {}
    for group in groups:
        runs = list(api.runs(PROJECT_PATH, filters={"group": group}))
        print(f"{group}: found {len(runs)} runs")
        for run in runs:
            run_by_id[run.id] = run
            rows.append(
                {
                    "group": group,
                    "run_id": run.id,
                    "run_name": run.name,
                    "state": run.state,
                    "created_at": run.created_at,
                    "url": run.url,
                }
            )

    runs_df = pd.DataFrame(rows)
    if runs_df.empty:
        raise ValueError("No W&B runs were found for the requested groups.")

    runs_df["group"] = pd.Categorical(runs_df["group"], categories=groups, ordered=True)
    return runs_df.sort_values(["group", "created_at", "run_id"]).reset_index(drop=True), run_by_id


def download_run_history(run: wandb.apis.public.Run, *, group: str | None = None, run_name: str | None = None) -> pd.DataFrame:
    rows = []
    for row in run.scan_history(page_size=1000):
        row = dict(row)
        step = row.get("_step", row.get("Step"))
        if step is None:
            continue
        row["raw_step"] = int(step)
        row["run_id"] = run.id
        row["run_name"] = run_name or run.name
        row["group"] = group or run.group
        rows.append(row)
    return pd.DataFrame(rows)


def download_run_histories(runs_df: pd.DataFrame, run_by_id: dict[str, wandb.apis.public.Run]) -> pd.DataFrame:
    histories = []
    for run_row in runs_df.itertuples(index=False):
        run = run_by_id[run_row.run_id]
        history = download_run_history(run, group=run_row.group, run_name=run_row.run_name)
        print(f"{run.id}: downloaded {len(history):,} history rows")
        if not history.empty:
            histories.append(history)

    initial_run = api.run(f"{PROJECT_PATH}/{INITIAL_RUN_ID}")
    initial_history = download_run_history(initial_run, group=INITIAL_RUN_NAME, run_name=INITIAL_RUN_NAME)
    print(f"{INITIAL_RUN_ID}: downloaded {len(initial_history):,} initial eval history rows")
    if not initial_history.empty:
        histories.append(initial_history)

    if not histories:
        raise ValueError("No W&B history rows were downloaded.")
    return pd.concat(histories, ignore_index=True, sort=False)


def extract_metrics(
    raw_history_df: pd.DataFrame,
    metrics: dict[str, str],
    *,
    min_step: int | None = STEP_BIN,
    max_step: int | None = MAX_STEP,
) -> pd.DataFrame:
    frames = []
    base_columns = ["group", "run_id", "run_name", "raw_step"]
    for metric_name, metric_key in metrics.items():
        if metric_key not in raw_history_df.columns:
            print(f"missing metric column: {metric_key}")
            continue
        metric_df = raw_history_df[base_columns + [metric_key]].dropna(subset=[metric_key]).copy()
        if metric_df.empty:
            print(f"no rows for metric column: {metric_key}")
            continue
        metric_df = metric_df.rename(columns={metric_key: "value"})
        metric_df["metric"] = metric_name
        metric_df["metric_key"] = metric_key
        metric_df["Step"] = metric_df["raw_step"].map(normalized_step)
        metric_df["value"] = metric_df["value"].astype(float)
        frames.append(metric_df)

    if not frames:
        raise ValueError("None of the requested metric columns were present in raw_history_df.")

    df = pd.concat(frames, ignore_index=True)
    if min_step is not None:
        df = df[df["Step"] >= min_step]
    if max_step is not None:
        df = df[df["Step"] <= max_step]
    df = df.copy()
    df = df.sort_values(["group", "run_id", "metric", "Step", "raw_step"])
    return df.groupby(["group", "run_id", "run_name", "metric", "metric_key", "Step"], as_index=False).agg(
        value=("value", "last"),
        raw_step=("raw_step", "last"),
    )


def extract_initial_eval(raw_history_df: pd.DataFrame, groups: list[str], metrics: dict[str, str]) -> pd.DataFrame:
    initial_history = raw_history_df[raw_history_df["run_id"] == INITIAL_RUN_ID].copy()
    extracted = extract_metrics(initial_history, metrics, min_step=None, max_step=None)
    latest_by_metric = extracted.sort_values("raw_step").groupby(["metric", "metric_key"], as_index=False).tail(1)
    rows = []
    for group in groups:
        for metric_row in latest_by_metric.itertuples(index=False):
            rows.append(
                {
                    "group": group,
                    "run_id": INITIAL_RUN_ID,
                    "run_name": INITIAL_RUN_NAME,
                    "metric": metric_row.metric,
                    "metric_key": metric_row.metric_key,
                    "Step": 0,
                    "value": metric_row.value,
                    "raw_step": 0,
                }
            )
    return pd.DataFrame(rows)

In [ ]:
runs_df, run_by_id = collect_group_runs(GROUPS)
raw_history_df = download_run_histories(runs_df, run_by_id)

display(runs_df)
print(f"Downloaded {len(raw_history_df):,} total history rows with {len(raw_history_df.columns):,} columns.")

## Extract eval/pass_at_1

In [ ]:
metrics_df = extract_metrics(raw_history_df[raw_history_df["run_id"] != INITIAL_RUN_ID], METRICS)
initial_df = extract_initial_eval(raw_history_df, GROUPS, METRICS)
plot_df = pd.concat([initial_df, metrics_df], ignore_index=True)
plot_df["group"] = pd.Categorical(plot_df["group"], categories=GROUPS, ordered=True)
plot_df = plot_df.sort_values(["group", "run_id", "metric", "Step"]).reset_index(drop=True)
pass_at_1_df = plot_df[plot_df["metric"] == PLOT_METRIC_NAME].copy()

display(plot_df.head())
display(
    pass_at_1_df.groupby(["group", "Step"], observed=True)
    .agg(mean_pass_at_1=("value", "mean"), std_pass_at_1=("value", "std"), n=("value", "size"))
    .reset_index()
    .head(16)
)

## Plot eval/pass_at_1

In [ ]:
sns.set_theme(
    context="paper",
    style="whitegrid",
    palette="Blues",
    font="DejaVu Sans",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
        "legend.fontsize": 8.5,
        "legend.title_fontsize": 9,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "grid.linewidth": 0.5,
        "lines.linewidth": 2.0,
    },
)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

fig, ax = plt.subplots(figsize=(5.25, 3.3), constrained_layout=True)
plot_ready_df = pass_at_1_df.copy()
plot_ready_df["setting"] = pd.Categorical(
    plot_ready_df["group"].map(GROUP_LABELS),
    categories=[GROUP_LABELS[group] for group in GROUPS],
    ordered=True,
)
blue_palette = dict(zip(plot_ready_df["setting"].cat.categories, sns.color_palette("Blues", n_colors=len(GROUPS) + 2)[2:]))

lineplot_kwargs = dict(
    data=plot_ready_df,
    x="Step",
    y="value",
    hue="setting",
    hue_order=plot_ready_df["setting"].cat.categories,
    palette=blue_palette,
    estimator="mean",
    err_style="band",
    ax=ax,
)

if tuple(int(part) for part in sns.__version__.split(".")[:2]) >= (0, 12):
    sns.lineplot(**lineplot_kwargs, errorbar="sd")
else:
    sns.lineplot(**lineplot_kwargs, ci="sd")

ax.set_title("GSM8K pass@1")
ax.set_xlabel("Step")
ax.set_ylabel("pass@1")
ax.set_xlim(0, MAX_STEP)
ax.set_xticks(range(0, MAX_STEP + STEP_BIN, 200))
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Setting", frameon=False, loc="best")
sns.despine(ax=ax)

fig.savefig(OUTPUT_DIR / "gsm8k_pass_at_1.pdf", bbox_inches="tight")

## Plot difficulty metrics

In [ ]:
DIFFICULTY_METRICS = {
    "easy": "eval/gsm8k_pass25/pass_at_1",
    "medium": "eval/gsm8k_pass10/pass_at_1",
    "hard": "eval/gsm8k_pass5/pass_at_1",
    "extra": "eval/gsm8k_pass0/pass_at_1",
}
DIFFICULTY_INITIAL_VALUES = {
    "easy": 25,
    "medium": 10,
    "hard": 5,
    "extra": 0,
}
DIFFICULTY_BASE_COLORS = {
    "easy": "#2ca25f",
    "medium": "#f0c419",
    "hard": "#f28e2b",
    "extra": "#d62728",
}


def blend_with_white(color: str, amount: float) -> tuple[float, float, float]:
    rgb = mpl.colors.to_rgb(color)
    return tuple((1 - amount) + amount * channel for channel in rgb)


difficulty_metrics_df = extract_metrics(raw_history_df[raw_history_df["run_id"] != INITIAL_RUN_ID], DIFFICULTY_METRICS)
difficulty_initial_df = pd.DataFrame(
    [
        {
            "group": group,
            "run_id": INITIAL_RUN_ID,
            "run_name": INITIAL_RUN_NAME,
            "metric": metric,
            "metric_key": DIFFICULTY_METRICS[metric],
            "Step": 0,
            "value": initial_value / 100,
            "raw_step": 0,
        }
        for group in GROUPS
        for metric, initial_value in DIFFICULTY_INITIAL_VALUES.items()
    ]
)
difficulty_plot_df = pd.concat([difficulty_initial_df, difficulty_metrics_df], ignore_index=True)
difficulty_plot_df["group"] = pd.Categorical(difficulty_plot_df["group"], categories=GROUPS, ordered=True)
difficulty_plot_df["setting"] = pd.Categorical(
    difficulty_plot_df["group"].map(GROUP_LABELS),
    categories=[GROUP_LABELS[group] for group in GROUPS],
    ordered=True,
)
difficulty_plot_df["metric"] = pd.Categorical(
    difficulty_plot_df["metric"],
    categories=list(DIFFICULTY_METRICS),
    ordered=True,
)
difficulty_plot_df["pass_at_1_percent"] = difficulty_plot_df["value"] * 100
difficulty_plot_df["series"] = difficulty_plot_df["metric"].astype(str) + " | " + difficulty_plot_df["setting"].astype(str)

shade_strengths = [0.45, 0.6, 0.78, 1.0]
difficulty_palette = {
    f"{metric} | {GROUP_LABELS[group]}": blend_with_white(DIFFICULTY_BASE_COLORS[metric], shade_strength)
    for metric in DIFFICULTY_METRICS
    for group, shade_strength in zip(GROUPS, shade_strengths)
}
series_order = [
    f"{metric} | {GROUP_LABELS[group]}"
    for metric in DIFFICULTY_METRICS
    for group in GROUPS
]

fig, ax = plt.subplots(figsize=(6.4, 4.0), constrained_layout=True)
lineplot_kwargs = dict(
    data=difficulty_plot_df,
    x="Step",
    y="pass_at_1_percent",
    hue="series",
    hue_order=series_order,
    palette=difficulty_palette,
    estimator="mean",
    err_style="band",
    ax=ax,
)

if tuple(int(part) for part in sns.__version__.split(".")[:2]) >= (0, 12):
    sns.lineplot(**lineplot_kwargs, errorbar="sd")
else:
    sns.lineplot(**lineplot_kwargs, ci="sd")

ax.set_title("GSM8K pass@1 by difficulty")
ax.set_xlabel("Step")
ax.set_ylabel("pass@1 (%)")
ax.set_xlim(0, MAX_STEP)
ax.set_xticks(range(0, MAX_STEP + STEP_BIN, 200))
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Difficulty | Setting", frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
sns.despine(ax=ax)

fig.savefig(OUTPUT_DIR / "gsm8k_pass_at_1_by_difficulty.pdf", bbox_inches="tight")